HUẤN LUYỆN VIT5-BASE TRÊN TẬP DỮ LIỆU vietnamese-summarization-dataset-0001  
SO SÁNH VỚI:
- ViT5 của Nyshikyen

In [27]:
# CẤU HÌNH CHẠY LOCAL TRONG VS CODE
from pathlib import Path
import sys

# Mở notebook từ thư mục gốc của dự án BTL_NLP.
# Ví dụ Windows: D:/homework/BTL_NLP
PROJECT_ROOT = Path.cwd().resolve()

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)


Python: d:\homework\BTL_NLP\.venv\Scripts\python.exe
Project root: D:\homework\BTL_NLP


In [28]:
from pathlib import Path

# Cấu trúc thư mục local tương tự trên Google Drive:
# BTL_NLP/
# ├── datasets/vietnamese-summarization-dataset-0001/
# ├── models/ViT5-base/
# ├── models/ViT5-base-finetuned-0001/
# └── training/vit5-base-0001-checkpoints/

LOCAL_MODEL = PROJECT_ROOT / "models" / "ViT5-base"
LOCAL_DATASET = (
    PROJECT_ROOT
    / "datasets"
    / "vietnamese-summarization-dataset-0001"
)

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "training"
    / "vit5-base-0001-checkpoints"
)

FINAL_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "ViT5-base-finetuned-0001"
)

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", LOCAL_MODEL)
print("Dataset:", LOCAL_DATASET)
print("Checkpoint:", CHECKPOINT_DIR)
print("Final model:", FINAL_MODEL_DIR)


Model: D:\homework\BTL_NLP\models\ViT5-base
Dataset: D:\homework\BTL_NLP\datasets\vietnamese-summarization-dataset-0001
Checkpoint: D:\homework\BTL_NLP\training\vit5-base-0001-checkpoints
Final model: D:\homework\BTL_NLP\models\ViT5-base-finetuned-0001


In [29]:
# Chạy một lần trong kernel Python của VS Code.
# Có thể bỏ qua cell này nếu môi trường ảo đã cài đủ thư viện.
# %pip install -q transformers datasets accelerate evaluate rouge-score sentencepiece protobuf


In [30]:
# Trên VS Code không cần copy dữ liệu như Colab.
# Model và dataset được đọc trực tiếp từ thư mục dự án local.

assert LOCAL_MODEL.exists(), (
    f"Không tìm thấy model tại: {LOCAL_MODEL}"
)

assert LOCAL_DATASET.exists(), (
    f"Không tìm thấy dataset tại: {LOCAL_DATASET}"
)

print("Đã tìm thấy model và dataset local.")


Đã tìm thấy model và dataset local.


**BẮT ĐẦU TRAIN**

In [31]:
from pathlib import Path
import torch

# Các đường dẫn đã được khai báo theo PROJECT_ROOT ở cell phía trên.
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

assert LOCAL_MODEL.exists(), (
    f"Không tìm thấy model tại: {LOCAL_MODEL}"
)

assert LOCAL_DATASET.exists(), (
    f"Không tìm thấy dataset tại: {LOCAL_DATASET}"
)

print("Model:", LOCAL_MODEL)
print("Dataset:", LOCAL_DATASET)
print("Checkpoint:", CHECKPOINT_DIR)
print("Final model:", FINAL_MODEL_DIR)

print("\nCUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2,
        ),
        "GB",
    )
else:
    print("Đang chạy bằng CPU.")


Model: D:\homework\BTL_NLP\models\ViT5-base
Dataset: D:\homework\BTL_NLP\datasets\vietnamese-summarization-dataset-0001
Checkpoint: D:\homework\BTL_NLP\training\vit5-base-0001-checkpoints
Final model: D:\homework\BTL_NLP\models\ViT5-base-finetuned-0001

CUDA: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.0 GB


In [32]:
from datasets import DatasetDict, load_dataset

data_files = {
    "train": str(LOCAL_DATASET / "train.jsonl"),
    "validation": str(LOCAL_DATASET / "validation.jsonl"),
    "test": str(LOCAL_DATASET / "test.jsonl"),
}

raw_dataset = load_dataset(
    "json",
    data_files=data_files,
)

print(raw_dataset)

print("\nCác cột:")
print(raw_dataset["train"].column_names)

print("\nMẫu train đầu tiên:")
print(raw_dataset["train"][0])


DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 15620
    })
    validation: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 1952
    })
    test: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 1953
    })
})

Các cột:
['document', 'summary', 'keywords']

Mẫu train đầu tiên:
{'document': 'Lá N của cây N lô hội N chứa V đầy A chất N gel N và bạn N có thể hái V mỗi khi N cần V . Nên V để khi N nào dùng V mới hái V . Cắt N một nhánh N từ cây lô hội N và vắt V hoặc múc V phần N gel V trong suốt A bên N trong N ra V . Nếu N thu hoạch V nhiều A , bạn N có thể cắt V đôi lá N cây N lô hội N ( theo V chiều dọc N ) để lấy V được hết V chất N gel V bên N trong N . Chỉ N nên hái V đủ A cho mỗi lần N sử dụng V . Nếu N còn thừa A , bạn N có thể đựng V trong hộp N có V nắp N đậy N kín A và trữ V trong tủ lạnh N được đến V một tuần N . Bạn V có thể bôi V trực tiếp A lô hội

In [33]:
TEXT_COLUMN = "document"
SUMMARY_COLUMN = "summary"


def valid_example(example):
    document = example.get(TEXT_COLUMN)
    summary = example.get(SUMMARY_COLUMN)

    return (
        isinstance(document, str)
        and isinstance(summary, str)
        and len(document.strip()) > 0
        and len(summary.strip()) > 0
    )


clean_dataset = DatasetDict()

for split_name, split_dataset in raw_dataset.items():
    before = len(split_dataset)

    filtered_dataset = split_dataset.filter(
        valid_example,
        desc=f"Cleaning {split_name}",
    )

    after = len(filtered_dataset)

    clean_dataset[split_name] = filtered_dataset

    print(
        f"{split_name}: "
        f"{after:,}/{before:,} mẫu hợp lệ"
    )

raw_dataset = clean_dataset

train: 15,620/15,620 mẫu hợp lệ
validation: 1,952/1,952 mẫu hợp lệ
test: 1,953/1,953 mẫu hợp lệ


In [34]:
DEBUG_MODE = False

DEBUG_TRAIN_SIZE = 500
DEBUG_VALIDATION_SIZE = 100
DEBUG_TEST_SIZE = 100

if DEBUG_MODE:
    dataset_for_training = DatasetDict({
        "train": raw_dataset["train"].select(
            range(
                min(
                    DEBUG_TRAIN_SIZE,
                    len(raw_dataset["train"]),
                )
            )
        ),
        "validation": raw_dataset[
            "validation"
        ].select(
            range(
                min(
                    DEBUG_VALIDATION_SIZE,
                    len(raw_dataset["validation"]),
                )
            )
        ),
        "test": raw_dataset["test"].select(
            range(
                min(
                    DEBUG_TEST_SIZE,
                    len(raw_dataset["test"]),
                )
            )
        ),
    })

    print("Đang chạy chế độ DEBUG")
else:
    dataset_for_training = raw_dataset

print(dataset_for_training)





# Mặc định KHÔNG xóa checkpoint.
# Chỉ đổi thành True khi thật sự muốn train lại từ đầu.
RESET_CHECKPOINTS = False

if RESET_CHECKPOINTS:
    import shutil

    if CHECKPOINT_DIR.exists():
        shutil.rmtree(CHECKPOINT_DIR)

    CHECKPOINT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("Đã chủ động xóa toàn bộ checkpoint.")
else:
    CHECKPOINT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("Giữ nguyên checkpoint hiện có.")

    existing_checkpoints = sorted(
        CHECKPOINT_DIR.glob("checkpoint-*")
    )

    if existing_checkpoints:
        print("Các checkpoint đang có:")

        for checkpoint_path in existing_checkpoints:
            print("-", checkpoint_path.name)
    else:
        print("Chưa có checkpoint nào.")

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 15620
    })
    validation: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 1952
    })
    test: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 1953
    })
})
Giữ nguyên checkpoint hiện có.
Các checkpoint đang có:
- checkpoint-116
- checkpoint-98


In [ ]:
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 192

print("Đang load tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    str(LOCAL_MODEL),
    use_fast=False,
    local_files_only=True,
)

print("Đang load model...")

model = AutoModelForSeq2SeqLM.from_pretrained(
    str(LOCAL_MODEL),
    local_files_only=True,
)

# Cần tắt cache khi dùng gradient checkpointing
model.config.use_cache = False

print(
    "Số tham số:",
    f"{model.num_parameters():,}",
)


def preprocess_function(examples):
    documents = [
        document.strip()
        for document in examples[TEXT_COLUMN]
    ]

    summaries = [
        summary.strip()
        for summary in examples[SUMMARY_COLUMN]
    ]

    model_inputs = tokenizer(
        documents,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=summaries,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


print("Đang tokenize dataset...")

tokenized_dataset = dataset_for_training.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_for_training[
        "train"
    ].column_names,
    desc="Tokenizing",
)

print(tokenized_dataset)

Đang load tokenizer...
Đang load model...


Loading weights: 100%|██████████| 257/257 [00:00<00:00, 6785.06it/s]

Số tham số: 225,950,976
Đang tokenize dataset...


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15620
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1952
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1953
    })
})


In [36]:
import evaluate
import numpy as np

rouge_metric = evaluate.load("rouge")


def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_predictions = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True,
    )

    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id,
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True,
    )

    decoded_predictions = [
        prediction.strip()
        for prediction in decoded_predictions
    ]

    decoded_labels = [
        label.strip()
        for label in decoded_labels
    ]

    result = rouge_metric.compute(
        predictions=decoded_predictions,
        references=decoded_labels,
        use_stemmer=False,
    )

    prediction_lengths = [
        np.count_nonzero(
            prediction != tokenizer.pad_token_id
        )
        for prediction in predictions
    ]

    result["gen_len"] = np.mean(
        prediction_lengths
    )

    return {
        key: round(float(value), 4)
        for key, value in result.items()
    }

In [37]:
import inspect

from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)

TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4

LEARNING_RATE = 3e-5
NUM_TRAIN_EPOCHS = 3

SAVE_STEPS = 250
EVAL_STEPS = 250
LOGGING_STEPS = 25

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)


def create_training_arguments():
    argument_names = inspect.signature(
        Seq2SeqTrainingArguments.__init__
    ).parameters

    use_bf16 = (
        torch.cuda.is_available()
        and hasattr(torch.cuda, "is_bf16_supported")
        and torch.cuda.is_bf16_supported()
    )

    training_kwargs = {
        "output_dir": str(CHECKPOINT_DIR),

        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": 0.01,
        "warmup_ratio": 0.05,
        "lr_scheduler_type": "linear",

        "per_device_train_batch_size":
            TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size":
            EVAL_BATCH_SIZE,
        "gradient_accumulation_steps":
            GRADIENT_ACCUMULATION_STEPS,

        "gradient_checkpointing": True,

        "fp16": (
            torch.cuda.is_available()
            and not use_bf16
        ),
        "bf16": use_bf16,

        "logging_strategy": "steps",
        "logging_steps": LOGGING_STEPS,

        "save_strategy": "steps",
        "save_steps": SAVE_STEPS,
        "save_total_limit": 2,

        "eval_steps": EVAL_STEPS,

        "predict_with_generate": True,
        "generation_max_length":
            MAX_TARGET_LENGTH,
        "generation_num_beams": 2,

        "load_best_model_at_end": True,
        "metric_for_best_model": "rougeL",
        "greater_is_better": True,

        "eval_accumulation_steps": 4,

        "report_to": "none",
        "push_to_hub": False,

        "seed": 42,
        "data_seed": 42,

        "dataloader_num_workers": 0,  # An toàn cho VS Code/Jupyter trên Windows
    }

    # Tương thích cả Transformers mới và cũ
    if "eval_strategy" in argument_names:
        training_kwargs["eval_strategy"] = "steps"
    else:
        training_kwargs[
            "evaluation_strategy"
        ] = "steps"

    return Seq2SeqTrainingArguments(
        **training_kwargs
    )


training_args = create_training_arguments()

trainer_kwargs = {
    "model": model,
    "args": training_args,

    "train_dataset":
        tokenized_dataset["train"],
    "eval_dataset":
        tokenized_dataset["validation"],

    "data_collator": data_collator,
    "compute_metrics": compute_metrics,

    "callbacks": [
        EarlyStoppingCallback(
            early_stopping_patience=3
        )
    ],
}

trainer_parameters = inspect.signature(
    Seq2SeqTrainer.__init__
).parameters

# Transformers mới dùng processing_class,
# phiên bản cũ dùng tokenizer
if "processing_class" in trainer_parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

print(training_args)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Seq2SeqTrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=4,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=250,
eval_strategy=steps,
eval_use_gather_objec

In [52]:
def find_latest_valid_checkpoint(
    checkpoint_root: Path,
):
    valid_checkpoints = []

    for path in checkpoint_root.glob(
        "checkpoint-*"
    ):
        if not path.is_dir():
            continue

        try:
            step = int(
                path.name.rsplit("-", 1)[-1]
            )
        except ValueError:
            continue

        has_model = any(
            (path / file_name).exists()
            for file_name in [
                "model.safetensors",
                "model.safetensors.index.json",
                "pytorch_model.bin",
                "pytorch_model.bin.index.json",
            ]
        )

        has_training_state = (
            (path / "trainer_state.json").exists()
            and (path / "optimizer.pt").exists()
            and (path / "scheduler.pt").exists()
        )

        if has_model and has_training_state:
            valid_checkpoints.append(
                (step, path)
            )

    if not valid_checkpoints:
        return None

    valid_checkpoints.sort(
        key=lambda item: item[0]
    )

    return str(valid_checkpoints[-1][1])


latest_checkpoint = (
    find_latest_valid_checkpoint(
        CHECKPOINT_DIR
    )
)

if latest_checkpoint:
    print(
        "Sẽ tiếp tục từ checkpoint:",
        latest_checkpoint,
    )
else:
    print(
        "Chưa có checkpoint, "
        "bắt đầu train từ đầu."
    )

print(CHECKPOINT_DIR.resolve())
import os

print("Current working directory:")
print(os.getcwd())

Sẽ tiếp tục từ checkpoint: D:\homework\BTL_NLP\training\vit5-base-0001-checkpoints\checkpoint-101
D:\homework\BTL_NLP\training\vit5-base-0001-checkpoints
Current working directory:
d:\homework\BTL_NLP


In [53]:
train_result = trainer.train(
    resume_from_checkpoint=latest_checkpoint
)

print("Train hoàn tất!")

print(train_result.metrics)

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
model.config.use_cache = True

trainer.save_model(
    str(FINAL_MODEL_DIR)
)

tokenizer.save_pretrained(
    str(FINAL_MODEL_DIR)
)

trainer.save_state()

trainer.log_metrics(
    "train",
    train_result.metrics,
)

trainer.save_metrics(
    "train",
    train_result.metrics,
)

print(
    "Đã lưu model cuối tại:",
    FINAL_MODEL_DIR,
)

In [ ]:
print("Đang đánh giá trên test...")

test_result = trainer.predict(
    tokenized_dataset["test"],
    metric_key_prefix="test",
)

trainer.log_metrics(
    "test",
    test_result.metrics,
)

trainer.save_metrics(
    "test",
    test_result.metrics,
)

print("\nKết quả test:")

for metric_name, value in (
    test_result.metrics.items()
):
    print(
        f"{metric_name:30s}: {value}"
    )

In [ ]:
import json

decoded_predictions = tokenizer.batch_decode(
    test_result.predictions,
    skip_special_tokens=True,
)

prediction_file = (
    FINAL_MODEL_DIR
    / "test_predictions.jsonl"
)

with prediction_file.open(
    "w",
    encoding="utf-8",
) as file:
    for index, prediction in enumerate(
        decoded_predictions
    ):
        original_example = (
            dataset_for_training["test"][index]
        )

        record = {
            "document":
                original_example[TEXT_COLUMN],
            "reference":
                original_example[SUMMARY_COLUMN],
            "prediction":
                prediction.strip(),
        }

        file.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

print(
    "Đã lưu dự đoán:",
    prediction_file,
)

In [54]:
import inspect

print("Global step hiện tại:", trainer.state.global_step)

save_function = trainer._save_checkpoint
parameters = inspect.signature(save_function).parameters

kwargs = {}

if "model" in parameters:
    kwargs["model"] = trainer.model

if "trial" in parameters:
    kwargs["trial"] = None

save_function(**kwargs)

print("Đã lưu checkpoint khẩn cấp tại:")
print(
    f"{trainer.args.output_dir}/"
    f"checkpoint-{trainer.state.global_step}"
)

Global step hiện tại: 250


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.57s/it]


Đã lưu checkpoint khẩn cấp tại:
D:\homework\BTL_NLP\training\vit5-base-0001-checkpoints/checkpoint-250


In [ ]:
# # Test thử model
# import torch
# from pathlib import Path
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# # ============================================================
# # 1. ĐƯỜNG DẪN MODEL ĐÃ LƯU
# # ============================================================
# MODEL_DIR = (
#     PROJECT_ROOT
#     / "models"
#     / "ViT5-base-finetuned-0001"
# )

# assert MODEL_DIR.exists(), f"Không tìm thấy model tại: {MODEL_DIR}"


# # ============================================================
# # 2. LOAD MODEL
# # ============================================================
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# tokenizer_test = AutoTokenizer.from_pretrained(
#     str(MODEL_DIR),
#     use_fast=False,
# )

# model_test = AutoModelForSeq2SeqLM.from_pretrained(
#     str(MODEL_DIR),
# )

# model_test.to(device)
# model_test.eval()

# print("Model:", MODEL_DIR)
# print("Device:", device)
# print("Tokenizer vocab:", len(tokenizer_test))


# # ============================================================
# # 3. THAM SỐ SINH TÓM TẮT
# # ============================================================
# MAX_INPUT_LENGTH = 512
# CHUNK_TOKEN_LENGTH = 450

# MAX_NEW_TOKENS = 150
# MIN_NEW_TOKENS = 30

# NUM_BEAMS = 4
# LENGTH_PENALTY = 1.0
# REPETITION_PENALTY = 1.15
# NO_REPEAT_NGRAM_SIZE = 3


# # ============================================================
# # 4. HÀM TÓM TẮT MỘT ĐOẠN
# # ============================================================
# @torch.inference_mode()
# def summarize_one_chunk(text: str) -> str:
#     text = text.strip()

#     if not text:
#         return ""

#     inputs = tokenizer_test(
#         text,
#         return_tensors="pt",
#         truncation=True,
#         max_length=MAX_INPUT_LENGTH,
#         padding=False,
#     )

#     inputs = {
#         key: value.to(device)
#         for key, value in inputs.items()
#     }

#     output_ids = model_test.generate(
#         **inputs,
#         max_new_tokens=MAX_NEW_TOKENS,
#         min_new_tokens=MIN_NEW_TOKENS,
#         num_beams=NUM_BEAMS,
#         do_sample=False,
#         early_stopping=True,
#         length_penalty=LENGTH_PENALTY,
#         repetition_penalty=REPETITION_PENALTY,
#         no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
#     )

#     summary = tokenizer_test.decode(
#         output_ids[0],
#         skip_special_tokens=True,
#         clean_up_tokenization_spaces=True,
#     )

#     return summary.strip()


# # ============================================================
# # 5. CHIA VĂN BẢN DÀI THEO TOKEN
# # ============================================================
# def split_text_by_tokens(
#     text: str,
#     max_tokens: int = CHUNK_TOKEN_LENGTH,
# ) -> list[str]:
#     token_ids = tokenizer_test.encode(
#         text,
#         add_special_tokens=False,
#     )

#     chunks = []

#     for start in range(0, len(token_ids), max_tokens):
#         chunk_ids = token_ids[start : start + max_tokens]

#         chunk_text = tokenizer_test.decode(
#             chunk_ids,
#             skip_special_tokens=True,
#             clean_up_tokenization_spaces=True,
#         )

#         if chunk_text.strip():
#             chunks.append(chunk_text.strip())

#     return chunks


# # ============================================================
# # 6. TÓM TẮT BÀI BÁO DÀI
# # ============================================================
# def summarize_article(text: str) -> dict:
#     text = text.strip()

#     if not text:
#         raise ValueError("Nội dung bài báo đang rỗng.")

#     input_token_count = len(
#         tokenizer_test.encode(
#             text,
#             add_special_tokens=False,
#         )
#     )

#     print(f"Số token đầu vào: {input_token_count}")

#     # Văn bản ngắn: tóm tắt trực tiếp
#     if input_token_count <= MAX_INPUT_LENGTH:
#         summary = summarize_one_chunk(text)

#         return {
#             "input_tokens": input_token_count,
#             "num_chunks": 1,
#             "chunk_summaries": [summary],
#             "final_summary": summary,
#         }

#     # Văn bản dài: tóm tắt từng phần
#     chunks = split_text_by_tokens(text)

#     print(f"Văn bản được chia thành {len(chunks)} đoạn.")

#     chunk_summaries = []

#     for index, chunk in enumerate(chunks, start=1):
#         print(f"Đang tóm tắt đoạn {index}/{len(chunks)}...")

#         chunk_summary = summarize_one_chunk(chunk)
#         chunk_summaries.append(chunk_summary)

#         print(f"Kết quả đoạn {index}:")
#         print(chunk_summary)
#         print("-" * 80)

#     # Nối các tóm tắt trung gian
#     combined_summary = "\n".join(chunk_summaries)

#     combined_token_count = len(
#         tokenizer_test.encode(
#             combined_summary,
#             add_special_tokens=False,
#         )
#     )

#     # Nếu phần nối vẫn dài, tiếp tục chia và tóm tắt
#     if combined_token_count > MAX_INPUT_LENGTH:
#         second_level_chunks = split_text_by_tokens(combined_summary)

#         second_level_summaries = [
#             summarize_one_chunk(chunk)
#             for chunk in second_level_chunks
#         ]

#         combined_summary = "\n".join(second_level_summaries)

#     # Tóm tắt lần cuối
#     final_summary = summarize_one_chunk(combined_summary)

#     return {
#         "input_tokens": input_token_count,
#         "num_chunks": len(chunks),
#         "chunk_summaries": chunk_summaries,
#         "final_summary": final_summary,
#     }


# # ============================================================
# # 7. DÁN BÀI BÁO CẦN TEST VÀO ĐÂY
# # ============================================================
# article_text = """
# Thành phố thông minh và hành trình chuyển đổi số bền vững.
# Trong những năm gần đây, nhiều địa phương đẩy mạnh chuyển đổi số
# trong quản lý đô thị, giao thông, y tế, giáo dục và môi trường. Các nền
# tảng dữ liệu dùng chung giúp cơ quan quản lý đưa ra quyết định nhanh
# hơn, trong khi người dân tiếp cận dịch vụ công thuận tiện hơn. Tuy
# nhiên, quá trình triển khai cũng đặt ra các thách thức về hạ tầng, an
# toàn thông tin, chi phí đầu tư, chất lượng dữ liệu và đào tạo nhân lực.
# Nhiều chuyên gia cho rằng thành công không chỉ phụ thuộc vào công nghệ
# mà còn đến từ sự phối hợp giữa chính quyền, doanh nghiệp, trường đại học
# và cộng đồng. Các chương trình thí điểm cho thấy việc ứng dụng trí tuệ
# nhân tạo để phân tích dữ liệu giao thông, dự báo ngập lụt, tối ưu năng
# lượng và hỗ trợ chăm sóc sức khỏe đã mang lại nhiều kết quả tích cực.
# Song song với đó, các cơ chế bảo vệ dữ liệu cá nhân, kiểm toán thuật
# toán và tiêu chuẩn mở được xem là nền tảng để phát triển lâu dài. Trong
# giai đoạn tiếp theo, mục tiêu hướng tới là xây dựng hệ sinh thái số bền
# vững, nơi mọi thành phần có thể chia sẻ dữ liệu an toàn, đổi mới sáng
# tạo và nâng cao chất lượng cuộc sống.
# """


# # ============================================================
# # 8. CHẠY TEST
# # ============================================================
# result = summarize_article(article_text)

# print("\n" + "=" * 90)
# print("TÓM TẮT CUỐI CÙNG")
# print("=" * 90)
# print(result["final_summary"])

# print("\nThông tin:")
# print("- Token đầu vào:", result["input_tokens"])
# print("- Số đoạn:", result["num_chunks"])


Loading weights: 100%|██████████| 257/257 [00:00<00:00, 2518.20it/s]
[transformers] Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=30) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model: D:\homework\BTL_NLP\models\ViT5-base-finetuned-0001
Device: cuda
Tokenizer vocab: 36096
Số token đầu vào: 457

TÓM TẮT CUỐI CÙNG
Bài viết này tập trung vào vấn đề an toàn thông tin và đào tạo nhân lực. Việc chuyển đổi số bền vững được xem là mục tiêu hướng tới để phát triển lâu dài. Tuy nhiên, các nền tảng dữ liệu có thể chung giúp quản lý đô thị, giao thông, y tế, giáo dục và môi trường đã mang lại nhiều kết quả tích cực. Song song với đó, việc cung cấp việc chăm sóc sức khỏe là rất quan trọng, đặc biệt là công nghệ.

Thông tin:
- Token đầu vào: 457
- Số đoạn: 1


In [51]:
# ============================================================
# TEST MODEL TRÊN TOÀN BỘ FILE test.jsonl
# Mỗi bài báo được ghi thành một file TXT riêng
# ============================================================

import json
import re
import time
from pathlib import Path

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ============================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN
# ============================================================

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "ViT5-base-finetuned-0001"
)

TEST_FILE = (
    PROJECT_ROOT
    / "datasets"
    / "vietnamese-summarization-dataset-0001"
    / "test.jsonl"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "vit5_test_predictions"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

assert MODEL_DIR.exists(), (
    f"Không tìm thấy model tại:\n{MODEL_DIR}"
)

assert TEST_FILE.exists(), (
    f"Không tìm thấy file test tại:\n{TEST_FILE}"
)

print("MODEL_DIR :", MODEL_DIR.resolve())
print("TEST_FILE :", TEST_FILE.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


# ============================================================
# 2. THAM SỐ CHẠY
# ============================================================

MAX_INPUT_LENGTH = 512
CHUNK_TOKEN_LENGTH = 450

MAX_NEW_TOKENS = 300
MIN_NEW_TOKENS = 40

NUM_BEAMS = 4
LENGTH_PENALTY = 1.2
REPETITION_PENALTY = 1.15
NO_REPEAT_NGRAM_SIZE = 3

# True: bỏ qua file đã sinh, tiếp tục từ bài chưa làm.
# False: chạy lại và ghi đè toàn bộ.
SKIP_EXISTING = True

# Đặt số nguyên để chỉ test một phần, ví dụ 20.
# Đặt None để chạy toàn bộ test.jsonl.
MAX_SAMPLES = None


# ============================================================
# 3. LOAD MODEL
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

tokenizer_test = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    use_fast=False,
)

model_test = AutoModelForSeq2SeqLM.from_pretrained(
    str(MODEL_DIR),
)

model_test.to(device)
model_test.eval()

print("\nModel:", MODEL_DIR)
print("Device:", device)
print("Tokenizer vocab:", len(tokenizer_test))

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )


# ============================================================
# 4. HÀM CHUẨN HÓA VĂN BẢN
# ============================================================

def normalize_text(value) -> str:
    """
    Chuyển dữ liệu về chuỗi và chuẩn hóa khoảng trắng.
    Không xóa dấu câu.
    """
    if value is None:
        return ""

    if isinstance(value, list):
        value = " ".join(
            str(item)
            for item in value
        )

    text = str(value)

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Gom nhiều khoảng trắng/xuống dòng thành một khoảng trắng
    text = " ".join(text.split())

    return text.strip()


def get_first_existing_value(
    record: dict,
    candidate_keys: list[str],
    default="",
):
    """
    Lấy giá trị đầu tiên từ các tên cột có thể tồn tại.
    """
    for key in candidate_keys:
        if key in record:
            value = record[key]

            if value is not None:
                return value

    return default


# ============================================================
# 5. TỰ NHẬN DIỆN CÁC TRƯỜNG TRONG JSONL
# ============================================================

GUID_KEYS = [
    "guid",
    "id",
    "doc_id",
    "document_id",
    "article_id",
    "news_id",
]

TITLE_KEYS = [
    "title",
    "headline",
    "heading",
    "name",
]

ARTICLE_KEYS = [
    "article",
    "text",
    "document",
    "content",
    "body",
    "news",
    "source",
    "input",
]

SUMMARY_KEYS = [
    "summary",
    "abstract",
    "reference_summary",
    "target",
    "label",
    "output",
]


def extract_record_fields(
    record: dict,
    index: int,
):
    guid = get_first_existing_value(
        record,
        GUID_KEYS,
        default=index,
    )

    title = get_first_existing_value(
        record,
        TITLE_KEYS,
        default="",
    )

    article = get_first_existing_value(
        record,
        ARTICLE_KEYS,
        default="",
    )

    reference_summary = get_first_existing_value(
        record,
        SUMMARY_KEYS,
        default="",
    )

    return {
        "index": index,
        "guid": normalize_text(guid),
        "title": normalize_text(title),
        "article": normalize_text(article),
        "reference_summary": normalize_text(
            reference_summary
        ),
    }


# ============================================================
# 6. ĐỌC FILE JSONL
# ============================================================

def load_jsonl(file_path: Path) -> list[dict]:
    records = []

    with file_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        for line_number, line in enumerate(
            file,
            start=1,
        ):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                print(
                    f"Bỏ qua dòng {line_number} "
                    f"do lỗi JSON: {error}"
                )
                continue

            records.append(record)

    return records


raw_records = load_jsonl(TEST_FILE)

if MAX_SAMPLES is not None:
    raw_records = raw_records[:MAX_SAMPLES]

print(
    "\nSố bài đọc được:",
    len(raw_records),
)

if not raw_records:
    raise ValueError(
        "File test.jsonl không có dữ liệu hợp lệ."
    )

print(
    "Các cột trong bản ghi đầu tiên:",
    list(raw_records[0].keys()),
)


# ============================================================
# 7. HÀM TÓM TẮT MỘT ĐOẠN
# ============================================================

@torch.inference_mode()
def summarize_one_chunk(
    text: str,
) -> str:
    text = normalize_text(text)

    if not text:
        return ""

    inputs = tokenizer_test(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
        padding=False,
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    output_ids = model_test.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        min_new_tokens=MIN_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        do_sample=False,
        early_stopping=False,
        length_penalty=LENGTH_PENALTY,
        repetition_penalty=REPETITION_PENALTY,
        no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
    )

    summary = tokenizer_test.decode(
        output_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    return normalize_text(summary)


# ============================================================
# 8. CHIA VĂN BẢN DÀI THEO TOKEN
# ============================================================

def split_text_by_tokens(
    text: str,
    max_tokens: int = CHUNK_TOKEN_LENGTH,
) -> list[str]:
    text = normalize_text(text)

    token_ids = tokenizer_test.encode(
        text,
        add_special_tokens=False,
    )

    chunks = []

    for start in range(
        0,
        len(token_ids),
        max_tokens,
    ):
        chunk_ids = token_ids[
            start:start + max_tokens
        ]

        chunk_text = tokenizer_test.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        chunk_text = normalize_text(chunk_text)

        if chunk_text:
            chunks.append(chunk_text)

    return chunks


# ============================================================
# 9. TÓM TẮT BÀI BÁO
# ============================================================

def summarize_article(
    text: str,
) -> dict:
    text = normalize_text(text)

    if not text:
        raise ValueError(
            "Nội dung bài báo đang rỗng."
        )

    input_token_ids = tokenizer_test.encode(
        text,
        add_special_tokens=False,
    )

    input_token_count = len(input_token_ids)

    # Văn bản không vượt giới hạn
    if input_token_count <= MAX_INPUT_LENGTH:
        final_summary = summarize_one_chunk(text)

        return {
            "input_tokens": input_token_count,
            "num_chunks": 1,
            "chunk_summaries": [
                final_summary
            ],
            "final_summary": final_summary,
        }

    # Văn bản dài: chia thành nhiều phần
    chunks = split_text_by_tokens(text)

    chunk_summaries = []

    for chunk in chunks:
        chunk_summary = summarize_one_chunk(
            chunk
        )

        if chunk_summary:
            chunk_summaries.append(
                chunk_summary
            )

    combined_summary = " ".join(
        chunk_summaries
    )

    combined_token_count = len(
        tokenizer_test.encode(
            combined_summary,
            add_special_tokens=False,
        )
    )

    # Nếu phần tóm tắt trung gian vẫn dài
    if combined_token_count > MAX_INPUT_LENGTH:
        second_level_chunks = (
            split_text_by_tokens(
                combined_summary
            )
        )

        second_level_summaries = []

        for chunk in second_level_chunks:
            summary = summarize_one_chunk(
                chunk
            )

            if summary:
                second_level_summaries.append(
                    summary
                )

        combined_summary = " ".join(
            second_level_summaries
        )

    final_summary = summarize_one_chunk(
        combined_summary
    )

    return {
        "input_tokens": input_token_count,
        "num_chunks": len(chunks),
        "chunk_summaries": chunk_summaries,
        "final_summary": final_summary,
    }


# ============================================================
# 10. TẠO TÊN FILE AN TOÀN
# ============================================================

def sanitize_filename(
    value: str,
    max_length: int = 80,
) -> str:
    value = normalize_text(value)

    # Xóa ký tự không hợp lệ trên Windows
    value = re.sub(
        r'[<>:"/\\|?*]',
        "_",
        value,
    )

    value = re.sub(
        r"\s+",
        "_",
        value,
    )

    value = value.strip("._ ")

    if not value:
        value = "untitled"

    return value[:max_length]


def make_output_path(
    index: int,
    guid: str,
    title: str,
) -> Path:
    safe_guid = sanitize_filename(
        guid,
        max_length=30,
    )

    safe_title = sanitize_filename(
        title,
        max_length=70,
    )

    file_name = (
        f"{index:05d}"
        f"_guid-{safe_guid}"
        f"_{safe_title}.txt"
    )

    return OUTPUT_DIR / file_name


# ============================================================
# 11. GHI KẾT QUẢ THEO ĐÚNG ĐỊNH DẠNG
# ============================================================

def save_result_txt(
    output_path: Path,
    index: int,
    guid: str,
    title: str,
    article: str,
    reference_summary: str,
    generated_summary: str,
    elapsed_seconds: float,
):
    content = (
        "INDEX:\n"
        f"{index}\n"
        "GUID:\n"
        f"{guid}\n"
        "TITLE:\n"
        f"{title}\n"
        "VĂN BẢN GỐC:\n"
        f"{article}\n"
        "TÓM TẮT THAM CHIẾU:\n"
        f"{reference_summary}\n"
        "TÓM TẮT DO VIT5 SINH:\n"
        f"{generated_summary}\n"
        "THỜI GIAN TÓM TẮT:\n"
        f"{elapsed_seconds:.4f} giây\n"
    )

    # Ghi file tạm trước, sau đó đổi tên.
    # Tránh file bị ghi dở nếu chương trình bị ngắt.
    temporary_path = output_path.with_suffix(
        ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        file.write(content)
        file.flush()

    temporary_path.replace(output_path)


# ============================================================
# 12. CHẠY TOÀN BỘ TEST DATASET
# ============================================================

success_count = 0
skip_count = 0
error_count = 0

error_log_path = (
    OUTPUT_DIR
    / "errors.log"
)

for index, raw_record in enumerate(
    tqdm(
        raw_records,
        desc="Đang tóm tắt",
    )
):
    fields = extract_record_fields(
        raw_record,
        index=index,
    )

    guid = fields["guid"]
    title = fields["title"]
    article = fields["article"]
    reference_summary = (
        fields["reference_summary"]
    )

    output_path = make_output_path(
        index=index,
        guid=guid,
        title=title,
    )

    # Nếu đã có kết quả thì bỏ qua
    if SKIP_EXISTING and output_path.exists():
        skip_count += 1
        continue

    if not article:
        error_count += 1

        with error_log_path.open(
            "a",
            encoding="utf-8",
        ) as error_file:
            error_file.write(
                f"INDEX {index}: "
                f"Không tìm thấy văn bản gốc. "
                f"Keys={list(raw_record.keys())}\n"
            )

        continue

    try:
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        result = summarize_article(
            article
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed_seconds = (
            time.perf_counter()
            - start_time
        )

        generated_summary = (
            result["final_summary"]
        )

        save_result_txt(
            output_path=output_path,
            index=index,
            guid=guid,
            title=title,
            article=article,
            reference_summary=reference_summary,
            generated_summary=generated_summary,
            elapsed_seconds=elapsed_seconds,
        )

        success_count += 1

        print(
            f"\n[{index + 1}/{len(raw_records)}] "
            f"Đã lưu: {output_path.name}"
        )
        print(
            f"Token: {result['input_tokens']} | "
            f"Chunks: {result['num_chunks']} | "
            f"Thời gian: {elapsed_seconds:.4f} giây"
        )

    except Exception as error:
        error_count += 1

        print(
            f"\nLỗi tại bài {index}:",
            repr(error),
        )

        with error_log_path.open(
            "a",
            encoding="utf-8",
        ) as error_file:
            error_file.write(
                f"INDEX {index}\n"
                f"GUID: {guid}\n"
                f"TITLE: {title}\n"
                f"ERROR: {repr(error)}\n"
                + "-" * 80
                + "\n"
            )

        # Giải phóng cache GPU sau lỗi
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================
# 13. THỐNG KÊ CUỐI
# ============================================================

print("\n" + "=" * 90)
print("HOÀN TẤT TEST MODEL")
print("=" * 90)
print("Tổng số bài      :", len(raw_records))
print("Đã tạo mới       :", success_count)
print("Đã có nên bỏ qua :", skip_count)
print("Bị lỗi           :", error_count)
print("Thư mục kết quả  :", OUTPUT_DIR.resolve())

if error_count > 0:
    print(
        "File ghi lỗi      :",
        error_log_path.resolve(),
    )

MODEL_DIR : D:\homework\BTL_NLP\models\ViT5-base-finetuned-0001
TEST_FILE : D:\homework\BTL_NLP\datasets\vietnamese-summarization-dataset-0001\test.jsonl
OUTPUT_DIR: D:\homework\BTL_NLP\results\vit5_test_predictions


Loading weights: 100%|██████████| 257/257 [00:00<00:00, 8641.19it/s]



Model: D:\homework\BTL_NLP\models\ViT5-base-finetuned-0001
Device: cuda
Tokenizer vocab: 36096
GPU: NVIDIA GeForce RTX 4050 Laptop GPU

Số bài đọc được: 1953
Các cột trong bản ghi đầu tiên: ['document', 'summary', 'keywords']


Đang tóm tắt:   0%|          | 0/1953 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=40) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=40) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentat


[1/1953] Đã lưu: 00000_guid-0_untitled.txt
Token: 1604 | Chunks: 4 | Thời gian: 32.6039 giây


...


[9/1953] Đã lưu: 00008_guid-8_untitled.txt
Token: 300 | Chunks: 1 | Thời gian: 4.8427 giây


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=40) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=40) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs


[10/1953] Đã lưu: 00009_guid-9_untitled.txt
Token: 719 | Chunks: 2 | Thời gian: 14.0684 giây


Đang tóm tắt:   1%|          | 10/1953 [02:52<9:19:47, 17.29s/it]


KeyboardInterrupt: 